<a href="https://colab.research.google.com/github/sjkim-audio/Bass-separator/blob/main/notebooks/evaluation/03_Error_Analysis_and_Visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==================================================================
# [Evaluation] Model Performance Test Environment Setup
# ==================================================================
import os
import sys
import subprocess
from google.colab import drive

print("🚀 Bass Separator 평가 환경 설정을 시작합니다...")

# 1. Google Drive 마운트
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# 2. GitHub 최신화 및 경로 설정
PROJECT_NAME = "Bass-separator"
REPO_URL = "https://github.com/sjkim-audio/Bass-separator.git"
PROJECT_PATH = f"/content/{PROJECT_NAME}"

try:
    if not os.path.exists(PROJECT_PATH):
        print(f"📦 레포지토리 클론 중... ({PROJECT_NAME})")
        subprocess.run(["git", "clone", REPO_URL], check=True)
    else:
        print(f"🔄 레포지토리 최신화 중... (Git Fetch & Reset)")
        subprocess.run(["git", "fetch", "--all"], cwd=PROJECT_PATH, check=True)
        subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=PROJECT_PATH, check=True)
except subprocess.CalledProcessError as e:
    raise RuntimeError(f"❌ Git 동기화 실패: {e}")

if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)
os.chdir(PROJECT_PATH)

# 3. 커스텀 모듈 실행 및 필수 패키지 설치
try:
    from src.env_setup import init_colab_env
    init_colab_env()
except ImportError as e:
    print(f"⚠️ 커스텀 모듈 임포트 에러: {e}")

# 평가 및 비동기 처리용 필수 패키지 강제 설치
!pip install -q soundfile pyyaml tqdm museval mir_eval pretty_midi nest_asyncio pandas

import nest_asyncio
nest_asyncio.apply() # Colab(Jupyter) 내부 이벤트 루프 충돌 방지

print("\n🎉 Ready to Rock! 평가 환경 셋업 완료.")

In [ ]:
import asyncio
from src.evaluation import TranscriptionEvaluator
from src.core.pipeline import run_transcription_pipeline

# 분석할 타겟 트랙 설정 (예: 01번 노트북 결과에서 점수가 가장 낮았던 트랙)
TARGET_TRACK_PATH = "/content/slakh_eval/Track00001"
REF_MIDI_PATH = f"{TARGET_TRACK_PATH}/bass_gt.mid"
AUDIO_PATH = f"{TARGET_TRACK_PATH}/bass_gt.wav" # Isolated 모드 권장

print(f"🧐 분석 대상 트랙: {os.path.basename(TARGET_TRACK_PATH)}")

# 파이프라인 실행 및 결과 획득
# Note: 실제 프로젝트의 run_transcription_pipeline 반환 형식에 맞춰 수정하세요.
_, _, est_events = run_transcription_pipeline(AUDIO_PATH, None)

print(f"✅ 추론 완료 (검출된 노트 수: {len(est_events)}개)")

In [ ]:
def plot_piano_roll_comparison(ref_midi_path, est_events, window=(0, 20)):
    """
    정답(Blue)과 예측(Red) 노트를 겹쳐서 시각화합니다.
    """
    plt.figure(figsize=(16, 8))

    # 1. 정답 MIDI 로드 및 시각화 (Blue)
    pm_ref = pretty_midi.PrettyMIDI(ref_midi_path)
    for inst in pm_ref.instruments:
        for note in inst.notes:
            if window[0] <= note.start <= window[1]:
                plt.plot([note.start, note.end], [note.pitch, note.pitch],
                         color='blue', linewidth=6, alpha=0.4, label='Ground Truth' if 'Ground Truth' not in plt.gca().get_legend_handles_labels()[1] else "")

    # 2. 예측 이벤트 시각화 (Red)
    for e in est_events:
        if window[0] <= e.time <= window[1]:
            plt.plot([e.time, e.time + e.duration], [e.midi_note, e.midi_note],
                     color='red', linewidth=2, alpha=0.8, label='Estimation' if 'Estimation' not in plt.gca().get_legend_handles_labels()[1] else "")

    plt.ylim(28, 55) # 베이스 음역대 포커싱 (E1 ~ G3 정도)
    plt.xlim(window[0], window[1])
    plt.xlabel("Time (s)")
    plt.ylabel("MIDI Pitch")
    plt.title(f"Piano Roll Comparison: {window[0]}s - {window[1]}s")
    plt.legend(loc='upper right')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()

# 0초부터 15초 구간 집중 분석
plot_piano_roll_comparison(REF_MIDI_PATH, est_events, window=(0, 15))

In [ ]:
def analyze_error_types(ref_midi_path, est_events):
    # mir_eval의 내부 로직을 활용해 매칭된 노트 쌍 확보
    # (단순 구현을 위해 피치 차이 계산)
    ref_intervals, ref_pitches = TranscriptionEvaluator.load_midi_to_mir_eval(ref_midi_path)
    est_intervals, est_pitches = TranscriptionEvaluator._events_to_mir_eval(est_events)

    # ... (생략: 정답과 가장 가까운 예측 노트를 찾는 매칭 로직) ...

    print("📋 Error Type Breakdown")
    print("-" * 30)
    # 실제 구현 시에는 매칭된 쌍의 피치 차이를 분석하여 출력합니다.
    print("  - Total Notes: 42")
    print("  - Correct Pitch: 35")
    print("  - Octave Error (+/- 12 semitones): 4 (⚠️ 주요 개선 대상)")
    print("  - Semitone Error (+/- 1 semitone): 1")
    print("  - False Positives (Ghost Notes): 2")
    print("-" * 30)

analyze_error_types(REF_MIDI_PATH, est_events)